# 7차시 학생용 실습 — 설비·공정 비교와 수율·불량 분석

오늘의 질문: **설비마다 차이가 있어 보이는데, 숫자로 계산해도 정말 그럴까?**

이번 차시는 Orange3로 먼저 훑어본 뒤 이어서 진행합니다. `# TODO` 부분만 채우면 됩니다.

## 1단계. 데이터 읽고 설비별 평균 온도 확인하기

### 따라 해보기 — 반별 평균 점수로 groupby 감 잡기
반도체 데이터로 들어가기 전에, 학생 6명을 반별로 묶어 평균 점수를 구하는 익숙한 예로 `groupby`의 동작부터
확인합니다(강의 자료 ② 절의 첫 코드 예제). 두 셀을 차례로 실행해 "① 같은 것끼리 묶기 → ② 묶음마다 계산하기"를 눈으로 확인해 보세요.

In [ ]:
# pandas를 불러오고, 학생 6명의 반과 점수가 담긴 작은 표를 직접 만들어 toy에 저장합니다.
# 실행하면 이름·반·점수 3개 열과 0~5번 행이 있는 표가 보입니다 — 반이 A, B, C 세 가지라는 점을 봐 두세요.
# pd.DataFrame({열 이름: [값 목록], ...}): 딕셔너리의 키가 열 이름, 리스트가 그 열의 값이 됩니다.
import pandas as pd

toy = pd.DataFrame({
    "이름": ["지민", "하은", "서준", "예린", "도윤", "시우"],
    "반": ["A", "A", "B", "B", "C", "C"],
    "점수": [90, 80, 70, 60, 85, 95],
})
print(toy)  # (중간 확인)

In [ ]:
# 같은 반끼리 묶은 뒤, 반마다 점수 평균을 계산합니다.
# 실행하면 A 85.0, B 65.0, C 90.0 세 줄이 나옵니다 — A반은 (90+80)/2=85.0처럼 계산된 값입니다.
# groupby("반"): 반 열의 값이 같은 행끼리 묶기만 합니다(이것만으로는 숫자가 나오지 않습니다).
# ["점수"].mean(): 묶인 그룹마다 점수 열의 평균을 계산합니다.
print(toy.groupby("반")["점수"].mean())

### 반도체 데이터에 적용하기 — 설비별 평균 온도
같은 방식을 오늘 다룰 반도체 데이터의 "설비별 평균 온도"에 적용합니다. 먼저 데이터를 읽어 모양을 확인한 뒤, 설비별로 묶어 평균을 냅니다.

In [ ]:
# 오늘 실습 CSV 파일(300행)을 읽어 df라는 표에 저장합니다. pd는 바로 위 셀에서 불러왔습니다.
# 실행하면 표의 앞 5행이 보입니다 — 로트번호부터 합격여부까지 열 11개를 훑어보세요. 불량유형 칸의 NaN은 "값 없음"(합격 웨이퍼라 불량유형이 비어 있음)이라는 뜻입니다.
df = pd.read_csv("../../data/weekly/week07/week07_equipment_yield.csv")
df.head()  # (중간 확인)

In [ ]:
# 앞에서 본 반별 평균과 같은 방식으로, 설비번호가 같은 행끼리 묶은 뒤 설비마다 온도_섭씨 평균을 계산합니다.
# 실행하면 EQ-01~EQ-04 네 설비의 평균 온도가 나옵니다 — 네 값이 모두 299~302 사이로 비슷한지 확인해 보세요.
# TODO: 설비별 평균 온도를 확인하세요.
print(df.____("설비번호")["온도_섭씨"].mean())

## 2단계. 설비별 합격률 비교하기

In [ ]:
# 설비별로 묶은 뒤, 설비마다 합격(1)인 웨이퍼의 비율(%)을 계산해 pass_rate에 저장하고 출력합니다.
# 실행하면 네 설비의 합격률이 나옵니다 — 평균 온도는 비슷했는데 합격률은 얼마나 차이 나는지 보세요.
# .apply(함수): 묶인 그룹마다 괄호 안의 함수를 적용합니다.
# lambda s: 식 — 그룹 s를 받아 뒤의 식을 계산해 돌려주는 이름 없는 짧은 함수입니다.
# (s == 1)은 합격이면 True, 아니면 False가 됩니다. True는 1, False는 0으로 계산되므로 이 값들의 평균이 곧 합격 비율입니다.
# TODO: 설비별 합격률(%)을 계산하세요.
pass_rate = df.groupby("설비번호")["합격여부"].apply(lambda s: (s == 1).____() * 100)
print(pass_rate)

## 3단계. 로트별 수율 계산하기

In [ ]:
# 로트번호가 같은 행(웨이퍼 4장)끼리 묶고, 로트마다 설비번호·검사수량·양품수량을 한 번에 계산해 lot_summary에 저장합니다.
# 실행하면 로트번호가 행 이름이 된 표의 앞 5행이 보입니다 — 로트마다 검사수량이 4인지 확인해 보세요.
# .agg(새 열 이름=("원본 열", 계산 방법), ...): 그룹마다 여러 계산을 한 번에 해서 새 열로 만듭니다.
# "first"는 그룹의 첫 번째 값, "count"는 개수, lambda는 직접 만든 계산(합격인 웨이퍼 수 세기)입니다.
lot_summary = df.groupby("로트번호").agg(
    설비번호=("설비번호", "first"),
    검사수량=("합격여부", "count"),
    양품수량=("합격여부", lambda s: (s == 1).sum()),
)
print(lot_summary.head())  # (중간 확인)

In [ ]:
# 로트마다 수율(%) = 양품 수 ÷ 검사한 수 × 100을 계산해 수율_pct라는 새 열로 추가하고, 앞 5행을 다시 봅니다.
# 실행하면 바로 위 표 오른쪽에 수율_pct 열이 하나 늘어난 표가 보입니다.
# 표["새 열 이름"] = 계산식: 표에 새 열을 만듭니다. .round(1)은 소수 첫째 자리까지 반올림합니다.
# TODO: 위 셀에서 집계한 표로 수율_pct를 계산하세요. (양품수량을 무엇으로 나눠야 할까요?)
lot_summary["수율_pct"] = (lot_summary["양품수량"] / lot_summary[____] * 100).round(1)

print(lot_summary.head())

## 4단계. 저수율 로트 찾기

In [ ]:
# 수율_pct가 80 미만인 로트만 골라 low_yield_lots에 저장하고, 그 로트가 몇 개인지 출력합니다.
# 실행하면 "__ 개 로트가 수율 80% 미만" 한 줄이 나옵니다. len(표)는 표의 행 개수입니다.
# TODO: 수율이 80 미만인 로트만 골라내세요.
low_yield_lots = lot_summary[lot_summary["수율_pct"] ____ 80]
print(len(low_yield_lots), "개 로트가 수율 80% 미만")

In [ ]:
# 저수율 로트들이 어느 설비에서 나왔는지 설비번호별로 개수를 셉니다.
# 실행하면 개수가 많은 설비부터 나옵니다 — 2단계에서 합격률이 가장 낮았던 설비가 맨 위에 있는지 비교해 보세요.
# .value_counts(): 열 안의 값마다 몇 번 나왔는지 세어 많은 순서로 보여줍니다.
print(low_yield_lots["설비번호"].value_counts())

## 5단계. 불량 유형 파레토

In [ ]:
# 불합격(-1)인 행만 골라낸 뒤, 불량유형 열의 값마다 몇 건인지 세어 defect_counts에 저장하고 출력합니다.
# 실행하면 가장 많은 불량유형부터 개수가 나옵니다 — 맨 위 유형이 가장 먼저 개선할 대상(파레토 관점)입니다.
# TODO: 불합격 행만 골라 불량유형별 개수를 세세요.
defect_counts = df[df["합격여부"] == ____]["불량유형"].value_counts()
print(defect_counts)

## 6단계(종합). 결론 정리하기

In [ ]:
# 2단계의 pass_rate에서 합격률이 가장 낮은 설비의 이름을 찾아 출력합니다.
# 실행하면 "합격률이 가장 낮은 설비: EQ-__" 한 줄이 나옵니다.
# .idxmin(): 가장 작은 값이 들어 있는 칸의 이름표(여기서는 설비번호)를 돌려줍니다.
# TODO: 합격률이 가장 낮은 설비를 idxmin()으로 찾아 출력하세요.
worst_equipment = pass_rate.____()
print(f"합격률이 가장 낮은 설비: {worst_equipment}")

In [ ]:
# 4단계의 저수율 로트를 설비별로 센 뒤, 그 개수가 가장 많은 설비의 이름을 출력합니다.
# 실행하면 "저수율 로트가 가장 많이 나온 설비: EQ-__" 한 줄이 나옵니다 — 바로 위 셀의 설비와 같은지 비교해 보세요.
# .idxmax(): idxmin과 반대로, 가장 큰 값이 들어 있는 칸의 이름표를 돌려줍니다.
print(f"저수율 로트가 가장 많이 나온 설비: {low_yield_lots['설비번호'].value_counts().idxmax()}")

## 7단계. 오늘의 학습을 한 문장으로 정리하기

> (예시) groupby로 설비별 평균·합격률을 비교하고, 로트별 수율을 따로 계산하면 전체 평균만으로는 보이지 않던 문제 로트를 찾을 수 있다.

### 도전 문제 1 — 설비×작업조 두 기준으로 묶기 (빠른 학습자용)
강의 자료 "도전 실습" 절의 도전 문제입니다. 2단계에서는 설비번호 하나로 묶었는데, 이번에는 설비번호와 작업조
**두 기준**으로 동시에 묶어 조합마다 합격률을 계산합니다. 특정 설비+작업조 조합이 유독 낮은 합격률을 보이는지
확인해 보세요. 위의 한 문장 정리를 마친 뒤 시간이 남으면 풀어 봅니다.

In [ ]:
# 설비번호와 작업조 두 기준으로 동시에 묶어, 설비×작업조 조합마다 합격률(%)을 계산합니다(강의 자료 "도전 실습"의 코드).
# 실행하면 설비번호 아래에 작업조가 들여쓰기된 두 겹의 표(11줄)가 나옵니다 — 유독 합격률이 낮은 조합이 있는지, 그런 조합이 한 설비에 몰려 있는지 찾아보세요.
# TODO: 설비번호와 작업조 두 열로 함께 묶고, 2단계와 같은 방법으로 합격률(%)을 계산하세요.
df.groupby(____)["합격여부"].apply(lambda s: (s==1).____()*100)

> **참고**
>
> - 첫 번째 빈칸(묶는 기준): 강의 자료 "도전 실습" 절의 도전 문제 상자에 이 코드가 그대로 있습니다. "주의할 점 & 자주 발생하는 오류" 절의 오류 2도 함께 보세요.
> - `groupby` 괄호 안에 기준 열 이름 **여러 개를 한 덩어리(리스트)로** 넣으면, 두 열의 값이 모두 같은 행끼리 묶입니다. 열 이름을 콤마로 따로 나열하면 두 번째 이름이 다른 뜻의 인자로 들어가 오류가 납니다.
> - 두 번째 빈칸(함수 이름): 강의 자료 "② 파이썬으로 확인하기 — groupby로 설비별 비교하기" 절의 "설비별 합격률 비교" 코드(노트북 2단계)에 같은 `lambda`가 있습니다. `(s==1)`은 합격이면 True(1), 아니면 False(0)이므로 이 값들의 평균이 곧 합격 비율이고, `*100`으로 %로 바꿉니다.

## 8단계. AI에게 질문하며 더 알아보기
오늘 배운 groupby 집계에 대해 AI 챗봇에게 질문해서 이해를 넓혀봅시다. 하나 이상 골라
물어보고, 새로 알게 된 내용을 다음 셀에 한두 줄로 정리해보세요.

- "groupby()와 pivot_table()은 어떻게 다른가요?"
- "파레토 법칙(80/20 법칙)이 정확히 무엇인가요?"
- "agg()에서 그룹마다 서로 다른 열에 서로 다른 함수를 적용하려면 어떻게 하나요?"
- "수율(yield)과 합격률은 같은 개념인가요, 다른 개념인가요?"

✏️ **내가 새로 알게 된 점**: (여기에 AI 답변을 요약해서 적어보세요)

## 9단계. AI에게 코드 생성 요청하고 직접 실행해보기
아래 프롬프트를 AI에게 그대로 물어보고, AI가 만들어준 코드를 다음 셀에 붙여넣어 실행해보세요.

- 프롬프트 예시: "설비별 합격률이 담긴 데이터를 받아서, 90% 미만인 설비만 경고 문구와 함께
  출력하는 코드를 만들어줘."

앞에서 계산한 `pass_rate`를 그대로 사용해도 됩니다.

**주의**: 강의 자료의 예시 코드는 합격률을 직접 적은 표에 `sample_rate`라는 새 이름을 붙여서, 2단계에서 계산한 `pass_rate`를 건드리지 않습니다. AI가 만든 코드가 `pass_rate = pd.Series({...})`처럼 같은 이름에 새 값을 넣는다면 이름을 바꿔 실행하세요. 이미 덮어썼다면 11단계 전에 2단계 셀을 다시 실행하면 됩니다 — 11단계가 2단계의 `pass_rate`를 그대로 씁니다.

In [ ]:
# 여기에 AI가 생성한 코드를 붙여넣고 실행해보세요.
# 실행하면 설비마다 정상 또는 점검 필요 문구가 붙은 결과가 나와야 합니다. 막히면 강의 자료의 "AI가 만들어줄 수 있는 코드 예시"를 붙여넣어도 됩니다.

## 10단계. 연습문제 — 불량 유형별 비율(%) 계산하기
5단계에서 계산한 `defect_counts`(불량유형별 개수)를 전체 대비 비율(%)로 바꿔봅니다.

In [ ]:
# 5단계의 defect_counts(불량유형별 개수)를 전체 불량 건수 합계로 나누고 100을 곱해 비율(%)로 바꿉니다.
# 실행하면 불량유형마다 전체 불량 중 몇 %인지 소수 첫째 자리까지 나옵니다 — 맨 위 유형의 비율을 확인해 보세요.
# TODO: defect_counts(불량유형별 개수)를 백분율로 변환하세요.
defect_pct = (defect_counts / defect_counts.____() * 100).round(1)
print(defect_pct)

## 11단계. AI로 재미있는 미니 프로그램 만들기 🎉 — 설비 랭킹 발표 프로그램
오늘 배운 groupby로 계산한 `pass_rate`(설비별 합격률)를 활용해, 순위를 매겨 시상식처럼
발표하는 프로그램을 AI에게 만들어달라고 요청해봅시다.

아래 프롬프트를 그대로 AI 챗봇(ChatGPT, Claude 등)에게 물어보고, 만들어준 코드를 다음 셀에
붙여넣어 실행해보세요.

**프롬프트 예시**: "설비 이름과 합격률이 담긴 데이터를 받아서, 합격률이 높은 순서로 정렬한 뒤
1~3등에게 🥇🥈🥉 메달 이모지를 붙여 시상식처럼 발표하는 파이썬 코드를 만들어줘."

**더 놀아보기**:
- 메달 대신 등수별 다른 메시지를 붙여달라고 요청해보거나, 꼴찌 설비에게는 "분발이 필요해요"
  같은 문구를 추가해달라고 해보세요.

In [ ]:
# 여기에 AI가 생성한 설비 랭킹 발표 코드를 붙여넣고 실행해보세요.
# 실행하면 합격률이 높은 설비부터 순위와 메달이 붙은 결과가 나와야 합니다. 2단계에서 계산한 pass_rate를 그대로 쓰면 됩니다.